# FrameWeave Hybrid v6.0 — TSO evidence-flow figure (Ovis2.5-9B)

[Open in Colab](https://colab.research.google.com/github/nqMinh-GR/FrameWeave/blob/main/ovis/notebooks/RelJoin_Residual_Ovis25_TSO_Evidence_Flow.ipynb)

Chỉ inference. Không train, không đổi prompt, không đổi thứ tự frame, không đổi pixel budget, không ghi đè checkpoint. Mỗi lần vẽ **một** câu Temporal Shot Ordering.

Mở notebook này từ GitHub hoặc Colab. Run All clone [FrameWeave](https://github.com/nqMinh-GR/FrameWeave.git) vào `/content/FrameWeave`, rồi symlink `reljoin/` và `run_*.py` vào `/content`. Không upload file source. Checkpoint epoch 5, CSV seed-42, upstream lite_v2 và jsonl seed 1/100/2027 nằm trong repo. Ovis2.5-9B và video HERBench lite tải từ Hugging Face. Cell cuối nén PDF, PNG, TeX và JSON rồi tải zip về máy.

`Q_ID` mặc định là `TSO_0916`. Câu sau: đặt `Q_ID = "TSO_1414"` và chạy lại cell extract. `DO_INSTALL = False` nên Run All không sửa paper.

Colab **A100**. Lần đầu cần khoảng 55 GB trống (model ~18 GB + video lite ~35 GB). Runtime còn nguyên thì lần sau bỏ qua phần đã tải.


## 1. Configuration


In [ ]:
# 1. Evidence-flow config — one HERBench TSO question / published Hybrid.pt
from __future__ import annotations

from pathlib import Path

assert Path("/content").exists(), "Notebook này chỉ chạy trên Google Colab (/content)."

ROOT = Path("/content")
Q_ID = "TSO_0916"          # câu tiếp theo: "TSO_1414"
ALLOW_ABSTENTION = False
DOWNLOAD_FULL_LITE = True  # Run All tải video lite nếu chưa có mp4
DO_INSTALL = False
METHOD_VERSION = "6.0.0"
EXPECTED_SHA256 = "3a19caac96f32e7bc82d5c373bf77c6b3884489ee2fb35c6957c5b7d8858da03"
CSV_NAME = "predictions_reljoin_01024e98f2b7c407.csv"
UPSTREAM_NAME = "multiseed_upstream_lite_v2.json"

REPO_URL = "https://github.com/nqMinh-GR/FrameWeave.git"
REPO_DIR = ROOT / "FrameWeave"
MODEL_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"
HERBENCH_ID = "DanBenAmi/HERBench"
HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"
HERBENCH_BASE_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/model_wrappers/base_vlm.py"
HERBENCH_ACC_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/calculate_accuracy.py"

VENV = ROOT / "ovis_env"
PYTHON = VENV / "bin" / "python"
WORKSPACE = ROOT / "fw"
PAPER = ROOT / "paper"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
EVAL_DIR = ROOT / "herbench_eval_pins"
ASSETS = REPO_DIR / "ovis" / "assets" / "evidence_flow"

print({
    "Q_ID": Q_ID,
    "ALLOW_ABSTENTION": ALLOW_ABSTENTION,
    "DOWNLOAD_FULL_LITE": DOWNLOAD_FULL_LITE,
    "DO_INSTALL": DO_INSTALL,
    "REPO_URL": REPO_URL,
    "WORKSPACE": str(WORKSPACE),
    "PAPER": str(PAPER),
    "MODEL_DIR": str(MODEL_DIR),
    "method": "FrameWeave-Hybrid",
    "method_version": METHOD_VERSION,
    "backbone": MODEL_ID,
    "checkpoint_sha256": EXPECTED_SHA256,
})


## 2. Environment (`ovis_env`)

Official Ovis2.5-9B pins: `torch==2.4.0`, `transformers==4.51.3`, `flash-attn==2.7.0.post2`.
Stop if flash-attn install fails — the ViT hard-imports it. Cell này cũng cài TeX để ghép TikZ.


In [ ]:
# 2. Bootstrap isolated ovis_env (Python 3.11 + torch 2.4.0 + flash-attn 2.7)
from __future__ import annotations

import os
import subprocess
from pathlib import Path
from shutil import which

assert Path("/content").exists(), "This notebook targets Google Colab (/content)."
ROOT = Path("/content")
VENV = ROOT / "ovis_env"
VENV_BIN = VENV / "bin"
PYTHON = VENV_BIN / "python"
PIP = VENV_BIN / "pip"
LOG = ROOT / "outputs_reljoin_v6_ovis" / "logs" / "cell2_evidence_flow_env.log"
LOG.parent.mkdir(parents=True, exist_ok=True)
LOG.write_text("", encoding="utf-8")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def sh(cmd, check=True, env=None):
    line = "$ " + " ".join(map(str, cmd))
    print(line, flush=True)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
        proc = subprocess.run(
            list(map(str, cmd)),
            check=False,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=env,
        )
        f.write(proc.stdout or "")
        f.write(f"\nexit={proc.returncode}\n\n")
    print(proc.stdout or "", end="")
    if check and proc.returncode != 0:
        print("LOG:", LOG)
        print((proc.stdout or "")[-4000:])
        raise RuntimeError(f"command failed ({proc.returncode}). See {LOG}")
    return proc


sh(["apt-get", "update", "-qq"], check=False)
sh([
    "apt-get", "install", "-y", "-qq",
    "python3.11", "python3.11-venv", "python3.11-dev", "ffmpeg",
    "texlive-latex-extra", "texlive-pictures", "texlive-fonts-recommended",
    "poppler-utils",
], check=False)
py311 = Path("/usr/bin/python3.11")
assert py311.exists(), "Không tìm thấy python3.11 — kiểm tra apt install."
assert which("pdflatex"), "pdflatex chưa có — cell 2 cần texlive-latex-extra."

need_recreate = True
if PYTHON.exists():
    ver = subprocess.check_output([str(PYTHON), "-V"], text=True).strip()
    print("Existing venv:", ver)
    need_recreate = "3.11" not in ver
if need_recreate:
    if VENV.exists():
        sh(["rm", "-rf", str(VENV)])
    sh([str(py311), "-m", "venv", "--copies", str(VENV)])
if not PIP.exists():
    sh([str(PYTHON), "-m", "ensurepip", "--upgrade"], check=False)
if not PIP.exists():
    get_pip = Path("/tmp/get-pip.py")
    sh(["curl", "-sSL", "https://bootstrap.pypa.io/get-pip.py", "-o", str(get_pip)])
    sh([str(PYTHON), str(get_pip)])
sh([str(PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools<70", "wheel"])
print("Python in venv:", subprocess.check_output([str(PYTHON), "-V"], text=True).strip())
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
    "torch==2.4.0", "torchvision==0.19.0",
    "--index-url", "https://download.pytorch.org/whl/cu121",
])
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed",
    "transformers==4.51.3", "numpy==1.25.0", "pillow==10.3.0", "moviepy==1.0.3",
    "huggingface_hub", "hf-transfer", "datasets", "accelerate", "einops", "sentencepiece",
    "opencv-python-headless", "pandas", "scipy", "tqdm", "ninja", "packaging",
])
print("Installing flash-attn==2.7.0.post2 (required by Ovis ViT)...")
fa_env = os.environ.copy()
fa_env["MAX_JOBS"] = "4"
sh(
    [str(PYTHON), "-m", "pip", "install", "flash-attn==2.7.0.post2", "--no-build-isolation"],
    env=fa_env,
)
check = subprocess.check_output([
    str(PYTHON), "-c",
    "import torch, transformers, flash_attn; "
    "print(torch.__version__); print(transformers.__version__); print(flash_attn.__version__); "
    "print(torch.__file__); print(transformers.__file__); "
    "print('cuda', torch.cuda.is_available())",
], text=True)
print(check)
for line in check.splitlines():
    if "/site-packages/" in line:
        assert "/content/ovis_env/" in line, f"Package leaked outside venv: {line}"
print("ovis_env ready:", PYTHON)
print("LOG:", LOG)


## 3. Drive + pin HERBench evaluators

Drive không bắt buộc. Mount lỗi thì Run All vẫn tiếp tục, vì checkpoint và log đã có trong repo. Không tải lại evaluator nếu SHA đã khớp.


In [ ]:
# 3. Drive is optional. Pin HERBench evaluator files used by the shared loader.
from __future__ import annotations

import hashlib
import os
import time
import urllib.request
from pathlib import Path

ROOT = Path("/content")
HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
OUT_DIR = ROOT / "outputs_reljoin_v6_ovis"
EVAL_DIR = ROOT / "herbench_eval_pins"
HERBENCH_BASE_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/model_wrappers/base_vlm.py"
HERBENCH_ACC_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/calculate_accuracy.py"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"
JSDELIVR_BASE = "https://cdn.jsdelivr.net/gh/DanBenAmi/HERBench@main/evaluation/model_wrappers/base_vlm.py"
JSDELIVR_ACC = "https://cdn.jsdelivr.net/gh/DanBenAmi/HERBench@main/evaluation/calculate_accuracy.py"

for p in [HERBENCH_ROOT, HERBENCH_VIDEOS, MODEL_DIR.parent, EVAL_DIR, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    RELJOIN_PERSIST = Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs")
except Exception as exc:
    print("Drive mount skipped:", exc)
    RELJOIN_PERSIST = OUT_DIR / "persist_local"
if RELJOIN_PERSIST.resolve() == OUT_DIR.resolve():
    RELJOIN_PERSIST = OUT_DIR / "persist_local"
    print("Persist dir was out_dir; using", RELJOIN_PERSIST)
RELJOIN_PERSIST.mkdir(parents=True, exist_ok=True)
os.environ["RELJOIN_PERSIST"] = str(RELJOIN_PERSIST)
os.environ["RELJOIN_ROOT"] = str(ROOT)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def fetch_pinned(urls: list[str], dest: Path, expected: str) -> None:
    if dest.exists() and sha256_file(dest) == expected:
        print("Already pinned:", dest.name)
        return
    last_err = None
    for url in urls:
        for attempt in range(4):
            try:
                print(f"GET {url} (try {attempt + 1})")
                urllib.request.urlretrieve(url, dest)
                got = sha256_file(dest)
                if got == expected:
                    print("OK", dest.name)
                    return
                last_err = RuntimeError(f"SHA mismatch for {dest.name}: {got}")
            except Exception as exc:
                last_err = exc
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Cannot pin {dest.name}: {last_err}")


fetch_pinned([HERBENCH_BASE_URL, JSDELIVR_BASE], EVAL_DIR / "base_vlm.py", HERBENCH_BASE_SHA256)
fetch_pinned([HERBENCH_ACC_URL, JSDELIVR_ACC], EVAL_DIR / "calculate_accuracy.py", HERBENCH_ACC_SHA256)
print({"RELJOIN_PERSIST": str(RELJOIN_PERSIST), "EVAL_DIR": str(EVAL_DIR)})


## 4. Clone source and verify

Run All clones the repo. Do not upload `.py` files. If `/content/reljoin` is a real folder from an older upload, restart the runtime before Run All.


In [ ]:
# Fetch FrameWeave source. Run All clones the repo; do not upload .py files.
from __future__ import annotations

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nqMinh-GR/FrameWeave.git"
REPO_DIR = Path("/content/FrameWeave")
ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])


def link(src: Path, dest: Path) -> None:
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        raise RuntimeError(
            f"{dest} already exists and is not the repo link. Restart the runtime, then Run All."
        )
    dest.symlink_to(src)


CODE = REPO_DIR / "ovis"
link(CODE / "reljoin", ROOT / "reljoin")
for script in sorted(CODE.glob("run_*.py")):
    link(script, ROOT / script.name)
print("FrameWeave source linked from", CODE)

required = [
    ROOT / "run_evidence_flow_figure.py",
    ROOT / "run_herbench_oracle_reljoin.py",
    ROOT / "reljoin" / "evidence_flow" / "signals.py",
    ROOT / "reljoin" / "evidence_flow" / "render.py",
    ROOT / "reljoin" / "evidence_flow" / "mine.py",
    ROOT / "reljoin" / "reljoin_modules.py",
    ROOT / "reljoin" / "ovis_hooks.py",
    ROOT / "reljoin" / "ovis_generate.py",
    ROOT / "reljoin" / "herbench_oracle.py",
    ROOT / "reljoin" / "tso_abstention.py",
    ROOT / "reljoin" / "frame_viz" / "geometry.py",
    ROOT / "reljoin" / "frame_viz" / "collector.py",
    CODE / "assets" / "evidence_flow" / "predictions_reljoin_01024e98f2b7c407.csv",
    CODE / "assets" / "evidence_flow" / "multiseed_upstream_lite_v2.json",
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, "Missing files — clone or symlink failed:\n" + "\n".join(missing)
print("Evidence-flow sources present.")
for p in required:
    print(f"  OK {p}")


## 5. Gắn checkpoint và log đã public

Ưu tiên `checkpoints/ovis/best_external_val/Hybrid.pt` trong repo đã clone. SHA phải là epoch 5. CSV, upstream và jsonl lấy từ `ovis/assets/evidence_flow/`. Drive chỉ được dùng khi file trong repo không có.


In [ ]:
# 5. Link the published Hybrid.pt, seed-42 CSV, upstream metadata, and seed jsonl.
from __future__ import annotations

import hashlib
from pathlib import Path

ROOT = Path("/content")
REPO_DIR = Path("/content/FrameWeave")
WORKSPACE = ROOT / "fw"
PAPER = ROOT / "paper"
ASSETS = REPO_DIR / "ovis" / "assets" / "evidence_flow"
EXPECTED_SHA256 = "3a19caac96f32e7bc82d5c373bf77c6b3884489ee2fb35c6957c5b7d8858da03"
CSV_NAME = "predictions_reljoin_01024e98f2b7c407.csv"
UPSTREAM_NAME = "multiseed_upstream_lite_v2.json"


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def ensure_link(src: Path, dest: Path) -> None:
    src = src.resolve()
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_symlink() or dest.exists():
        if dest.resolve() == src:
            print("already linked", dest)
            return
        if dest.is_dir() and not dest.is_symlink():
            raise RuntimeError(f"{dest} is a directory, not a file link")
        dest.unlink()
    dest.symlink_to(src)
    print(f"link {dest} -> {src}")


candidates = [
    REPO_DIR / "checkpoints" / "ovis" / "best_external_val" / "Hybrid.pt",
    ROOT / "outputs_reljoin_v6_ovis" / "checkpoints" / "best_external_val" / "Hybrid.pt",
    Path("/content/drive/MyDrive/outputs_reljoin_v6_ovis/checkpoints/best_external_val/Hybrid.pt"),
    Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs/outputs_reljoin_v6_ovis/checkpoints/best_external_val/Hybrid.pt"),
    Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs/checkpoints/best_external_val/Hybrid.pt"),
]
matched = None
for path in candidates:
    if not path.is_file():
        print("missing", path)
        continue
    digest = sha256_file(path)
    ok = digest == EXPECTED_SHA256
    print(f"{'OK' if ok else 'SKIP'} {digest} {path}")
    if ok and matched is None:
        matched = path
assert matched is not None, (
    "Không thấy Hybrid.pt epoch 5. Repo phải có checkpoints/ovis/best_external_val/Hybrid.pt."
)
ckpt_dest = (
    WORKSPACE / "v6_ovis" / "outputs_reljoin_v6_ovis" / "checkpoints" / "best_external_val" / "Hybrid.pt"
)
ensure_link(matched, ckpt_dest)
ensure_link(matched, ROOT / "outputs_reljoin_v6_ovis" / "checkpoints" / "best_external_val" / "Hybrid.pt")

csv_src = ASSETS / CSV_NAME
upstream_src = ASSETS / UPSTREAM_NAME
assert csv_src.is_file(), f"Thiếu {csv_src}"
assert upstream_src.is_file(), f"Thiếu {upstream_src}"
ensure_link(
    csv_src,
    WORKSPACE / "v6_ovis" / "reljoin_v6_ovis_outputs" / "eval_herbench_v2lite" / CSV_NAME,
)
ensure_link(upstream_src, PAPER / "audit" / UPSTREAM_NAME)

linked_seeds = []
for seed in (1, 100, 2027):
    src = ASSETS / f"seed_{seed}" / "predictions.jsonl"
    if not src.is_file():
        print(f"seed {seed} jsonl not in the repo")
        continue
    dest = (
        WORKSPACE / "v6_ovis_ablation"
        / "frameweave_ovis_multiseed_outputs-20261018T091434Z-1-001"
        / "frameweave_ovis_multiseed_outputs"
        / f"seed_{seed}" / "eval" / f"seed_{seed}" / "predictions.jsonl"
    )
    ensure_link(src, dest)
    linked_seeds.append(seed)

print({
    "Hybrid.pt": str(matched),
    "csv": str(csv_src),
    "upstream": str(upstream_src),
    "seeds_linked": linked_seeds,
})


## 6. Tải Ovis2.5-9B + HERBench lite

`DOWNLOAD_FULL_LITE = True` nên Run All tải video lite khi chưa đủ mp4. PerceptionTest không tải.


In [ ]:
# 6. Download Ovis2.5-9B and, when needed, the HERBench lite videos.
from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
assert PYTHON.exists(), "Chạy cell 2 trước"
os.environ["PATH"] = f"{ROOT / 'ovis_env' / 'bin'}:{os.environ.get('PATH', '')}"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
HERBENCH_HF_ID = "DanBenAmi/HERBench"
HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
MODEL_HF_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"

for p in [HERBENCH_ROOT, HERBENCH_VIDEOS, MODEL_DIR.parent]:
    p.mkdir(parents=True, exist_ok=True)


def _from_cell1(name, default):
    try:
        ip = get_ipython()
    except NameError:
        ip = None
    ns = ip.user_ns if ip is not None else {}
    return ns[name] if name in ns else default


DOWNLOAD_FULL_LITE = bool(_from_cell1("DOWNLOAD_FULL_LITE", True))

print("Downloading Ovis2.5-9B (~18 GB bf16)...")
subprocess.run([str(PYTHON), "-c", f"""
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='{MODEL_HF_ID}',
    revision='{MODEL_REVISION}',
    local_dir=r'{MODEL_DIR}',
    ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
)
print('model OK')
"""], check=True)
assert (MODEL_DIR / "config.json").exists(), f"Missing {MODEL_DIR / 'config.json'}"

LITE_MP4_MIN = 68
n_mp4 = len(list(HERBENCH_VIDEOS.rglob("*.mp4"))) if HERBENCH_VIDEOS.exists() else 0
if DOWNLOAD_FULL_LITE and n_mp4 < LITE_MP4_MIN:
    print("Downloading HERBench lite (~35GB)...")
    subprocess.run([str(PYTHON), "-c", f"""
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='{HERBENCH_HF_ID}',
    repo_type='dataset',
    revision='{HERBENCH_REVISION}',
    local_dir=r'{HERBENCH_ROOT}',
    allow_patterns=[
        'data/herbench_lite_v2.parquet',
        'data/herbench_annotations_lite_v2.json',
        'data/*metadata.json',
        'videos/videos.tar.part.00',
        'videos/videos.tar.part.01',
        'videos/videos.tar.part.02',
        'videos/videos.tar.part.03',
        'videos/videos_lite_info.txt',
        'videos/videos.tar.checksums.txt',
    ],
)
print('herbench snapshot OK')
"""], check=True)
    parts = sorted(HERBENCH_VIDEOS.glob("videos.tar.part.*"))
    print("parts:", [p.name for p in parts])
    mp4s = list(HERBENCH_VIDEOS.rglob("*.mp4"))
    assert len(parts) >= 4 or len(mp4s) >= LITE_MP4_MIN, (
        f"Thiếu videos.tar.part.00–03 trong {HERBENCH_VIDEOS}. Có: {[p.name for p in parts]}"
    )
    if len(mp4s) < LITE_MP4_MIN:
        cat = subprocess.Popen(["cat", *map(str, parts)], stdout=subprocess.PIPE)
        subprocess.run(["tar", "-xf", "-", "-C", str(HERBENCH_VIDEOS)], stdin=cat.stdout, check=True)
        cat.wait()
        print("Extracted archives")
    n_mp4 = len(list(HERBENCH_VIDEOS.rglob("*.mp4")))
    if n_mp4 >= LITE_MP4_MIN:
        for part in parts:
            part.unlink(missing_ok=True)
        print("Removed archives after full extract")
else:
    print(f"Skip lite download. videos_mp4={n_mp4} DOWNLOAD_FULL_LITE={DOWNLOAD_FULL_LITE}")

print({
    "videos_mp4": n_mp4,
    "model_config": (MODEL_DIR / "config.json").exists(),
    "free_GiB": round(shutil.disk_usage(ROOT).free / 2**30, 2),
})
assert (MODEL_DIR / "config.json").exists(), "Ovis config.json missing — tải lại cell 6."
assert n_mp4 >= 1, "Không có mp4. Bật DOWNLOAD_FULL_LITE hoặc giải nén lite trước."
print("Downloads OK")


## 7. Rank TSO và CPU checks

Rank không load Ovis. Check chạy self-test merge-order và ablation trên tensor giả. Layout smoke trong `/tmp` là hình synthetic, không phải kết quả paper.


In [ ]:
# 7. Rank the 152 TSO questions, then run the CPU checks. No heatmap.
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
WORKSPACE = ROOT / "fw"
PAPER = ROOT / "paper"
RUNNER = ROOT / "run_evidence_flow_figure.py"
assert PYTHON.exists(), "Chạy cell 2 trước — ovis_env missing."
assert RUNNER.is_file(), "Chạy cell 4 trước — source chưa được clone."
assert (PAPER / "audit" / "multiseed_upstream_lite_v2.json").is_file(), "Chạy cell 5 trước."

env = os.environ.copy()
env.pop("PYTHONPATH", None)
env.pop("PYTHONHOME", None)
env["PYTHONNOUSERSITE"] = "1"
env["PATH"] = f"{ROOT / 'ovis_env' / 'bin'}:{env.get('PATH', '')}"
env["TOKENIZERS_PARALLELISM"] = "false"

rank_cmd = [
    str(PYTHON), "-u", str(RUNNER),
    "--workspace", str(WORKSPACE),
    "--paper", str(PAPER),
    "rank",
]
print("$", " ".join(rank_cmd), flush=True)
subprocess.run(rank_cmd, check=True, env=env, cwd=str(ROOT))
report = json.loads((PAPER / "audit" / "qual_candidates_ranked.json").read_text(encoding="utf-8"))
print({
    "n_tso": report.get("n_tso"),
    "n_concrete_wrong": report.get("n_concrete_wrong"),
    "render_queue": report.get("render_queue"),
    "checkpoint_sha256_matches": report.get("checkpoint_sha256_matches"),
})
for row in report.get("concrete_wrong") or []:
    print(
        row["question_id"],
        "vanilla", row["vanilla_letter"],
        "gold", row["gold"],
        "seeds", row["n_seeds_correct"],
        "eligible", row["eligible_for_render"],
    )

check_cmd = [
    str(PYTHON), "-u", "-c",
    "from reljoin.evidence_flow.checks import run_intervention_checks, run_static_checks\n"
    "run_static_checks(None)\n"
    "run_intervention_checks()\n",
]
print("$ evidence-flow checks", flush=True)
subprocess.run(check_cmd, check=True, env=env, cwd=str(ROOT))
print("Rank and CPU checks finished. GPU extract is still not_run.")


## 8. Extract đúng một câu

Cell này load Ovis, chạy impulse test, rồi forward câu `Q_ID`. Exit code 2 nghĩa là thiếu CUDA, weight hoặc video: không có heatmap. Đổi `Q_ID` trong cell 1 và chạy lại cell này cho câu kế tiếp.


In [ ]:
# 8. Extract one TSO question. Re-run after changing Q_ID in cell 1.
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
WORKSPACE = ROOT / "fw"
PAPER = ROOT / "paper"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
HERBENCH_VIDEOS = ROOT / "HERBench_official" / "videos"
RUNNER = ROOT / "run_evidence_flow_figure.py"


def _from_cell1(name, default):
    try:
        ip = get_ipython()
    except NameError:
        ip = None
    ns = ip.user_ns if ip is not None else {}
    return ns[name] if name in ns else default


Q_ID = str(_from_cell1("Q_ID", "TSO_0916") or "TSO_0916")
ALLOW_ABSTENTION = bool(_from_cell1("ALLOW_ABSTENTION", False))
CKPT = WORKSPACE / "v6_ovis" / "outputs_reljoin_v6_ovis" / "checkpoints" / "best_external_val" / "Hybrid.pt"
report_path = PAPER / "audit" / "qual_candidates_ranked.json"
assert PYTHON.exists() and RUNNER.is_file() and CKPT.is_file()
assert (MODEL_DIR / "config.json").is_file(), "Chạy cell 6 trước — missing Ovis config.json"
assert report_path.is_file(), "Chạy cell 7 trước — chưa có qual_candidates_ranked.json"

report = json.loads(report_path.read_text(encoding="utf-8"))
pools = list(report.get("concrete_wrong") or [])
if ALLOW_ABSTENTION:
    pools.extend(report.get("abstention_flips_top10") or [])
case = next((row for row in pools if row["question_id"] == Q_ID), None)
assert case is not None, f"{Q_ID} không nằm trong hàng concrete-wrong."
rel = Path(case["video_path"])
video_candidates = [
    ROOT / "HERBench_official" / rel,
    HERBENCH_VIDEOS / Path(*rel.parts[1:]) if rel.parts and rel.parts[0] == "videos" else HERBENCH_VIDEOS / rel,
    HERBENCH_VIDEOS / rel.name,
]
video = next((path for path in video_candidates if path.is_file()), None)
assert video is not None, (
    f"Không thấy video của {Q_ID}: {case['video_path']}. "
    f"Đã tìm: {[str(path) for path in video_candidates]}"
)
print({"Q_ID": Q_ID, "video": str(video), "vanilla": case["vanilla_letter"], "gold": case["gold"]})

env = os.environ.copy()
env.pop("PYTHONPATH", None)
env.pop("PYTHONHOME", None)
env["PYTHONNOUSERSITE"] = "1"
env["PATH"] = f"{ROOT / 'ovis_env' / 'bin'}:{env.get('PATH', '')}"
env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"
cmd = [
    str(PYTHON), "-u", str(RUNNER),
    "--workspace", str(WORKSPACE),
    "--paper", str(PAPER),
    "extract",
    "--model-dir", str(MODEL_DIR),
    "--videos-root", str(HERBENCH_VIDEOS),
    "--checkpoint", str(CKPT),
    "--q-id", Q_ID,
    "--root", "/tmp/evidence_flow_root",
]
if ALLOW_ABSTENTION:
    cmd.append("--allow-abstention")
print("$", " ".join(cmd), flush=True)
proc = subprocess.run(cmd, check=False, env=env, cwd=str(ROOT))
status_path = PAPER / "audit" / f"qual_evidence_flow_{Q_ID}.status.json"
audit_path = PAPER / "audit" / f"qual_evidence_flow_{Q_ID}.json"
pdf_path = PAPER / "figures" / "qual_evidence_flow" / f"{Q_ID}.pdf"
png_path = pdf_path.with_suffix(".png")
print({"returncode": proc.returncode, "pdf_exists": pdf_path.is_file()})
if proc.returncode == 2 and status_path.is_file():
    print(status_path.read_text(encoding="utf-8"))
elif proc.returncode != 0:
    raise RuntimeError(f"extract failed with exit {proc.returncode}")
elif audit_path.is_file():
    bundle = json.loads(audit_path.read_text(encoding="utf-8"))
    print({
        "render_allowed": bundle.get("render_allowed"),
        "reject_reason": bundle.get("reject_reason"),
        "chosen_k": bundle.get("chosen_k"),
        "overall_delta_z": bundle.get("overall_delta_z"),
        "overall_delta_margin": bundle.get("overall_delta_margin"),
    })
if png_path.is_file():
    from IPython.display import Image, display
    display(Image(filename=str(png_path)))
elif not pdf_path.is_file():
    print("No paper PDF for", Q_ID)


## 9. Install vào paper

Mặc định tắt. Run All không sửa `fig:qual_main`.


In [ ]:
# 9. Install stays off unless DO_INSTALL is True.
from __future__ import annotations

import os
import subprocess
from pathlib import Path


def _from_cell1(name, default):
    try:
        ip = get_ipython()
    except NameError:
        ip = None
    ns = ip.user_ns if ip is not None else {}
    return ns[name] if name in ns else default


DO_INSTALL = bool(_from_cell1("DO_INSTALL", False))
Q_ID = str(_from_cell1("Q_ID", "TSO_0916") or "TSO_0916")
if not DO_INSTALL:
    print("DO_INSTALL=False — không sửa fig:qual_main.")
else:
    ROOT = Path("/content")
    PAPER = ROOT / "paper"
    RUNNER = ROOT / "run_evidence_flow_figure.py"
    assert (PAPER / "main.tex").is_file(), f"Thiếu {PAPER / 'main.tex'}"
    assert (PAPER / "figures" / "qual_evidence_flow" / f"{Q_ID}.pdf").is_file()
    env = os.environ.copy()
    env.pop("PYTHONPATH", None)
    env.pop("PYTHONHOME", None)
    env["PYTHONNOUSERSITE"] = "1"
    cmd = [
        str(ROOT / "ovis_env" / "bin" / "python"), "-u", str(RUNNER),
        "--workspace", str(ROOT / "fw"),
        "--paper", str(PAPER),
        "install",
        "--q-id", Q_ID,
    ]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, env=env, cwd=str(ROOT))


## 10. Zip và tải về máy

Zip chứa PDF, PNG, TeX, panel và JSON của câu vừa chạy. Không có PDF thì zip vẫn chứa JSON lý do dừng.


In [ ]:
# 10. Zip this question's artifacts and download them. Does not touch eval CSVs.
from __future__ import annotations

import shutil
from pathlib import Path


def _from_cell1(name, default):
    try:
        ip = get_ipython()
    except NameError:
        ip = None
    ns = ip.user_ns if ip is not None else {}
    return ns[name] if name in ns else default


Q_ID = str(_from_cell1("Q_ID", "TSO_0916") or "TSO_0916")
PAPER = Path("/content/paper")
stage = Path("/tmp") / f"qual_evidence_flow_{Q_ID}"
if stage.exists():
    shutil.rmtree(stage)
stage.mkdir(parents=True)
names = [
    PAPER / "audit" / "qual_candidates_ranked.json",
    PAPER / "audit" / f"qual_evidence_flow_{Q_ID}.json",
    PAPER / "audit" / f"qual_evidence_flow_{Q_ID}.status.json",
    PAPER / "fig" / f"qual_evidence_flow_{Q_ID}.tex",
    PAPER / "figures" / "qual_evidence_flow" / f"{Q_ID}.pdf",
    PAPER / "figures" / "qual_evidence_flow" / f"{Q_ID}.png",
]
copied = []
for src in names:
    if src.is_file():
        dest = stage / src.name
        shutil.copy2(src, dest)
        copied.append(dest.name)
panels = PAPER / "figures" / "qual_evidence_flow" / f"{Q_ID}_panels"
if panels.is_dir():
    shutil.copytree(panels, stage / panels.name)
    copied.append(panels.name + "/")
assert copied, f"Không có artifact để nén cho {Q_ID}. Chạy cell extract trước."
archive = shutil.make_archive(str(Path("/content") / f"qual_evidence_flow_{Q_ID}"), "zip", root_dir=stage)
print(archive, copied)
try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print("Download skipped:", exc)
